# PREPROCESSING TEKS ULASAN E-COMMERCE

**Nama:** Fitri Simah Bengi

**NIM:** 25146043

**Mata Kuliah:** Natural Language Processing

In [1]:
import pandas as pd
import numpy as np
import re
import requests

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("Library berhasil dipanggil!")

Library berhasil dipanggil!


In [2]:
url = "https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment/resolve/main/labeledReview.datasetFix.json"

response = requests.get(url)

print("Status:", response.status_code)

Status: 200


In [3]:
data = response.json()

df = pd.DataFrame(data)

print("Jumlah baris dan kolom:", df.shape)
print("\nNama kolom:")
print(df.columns.tolist())

display(df.head())

Jumlah baris dan kolom: (11606, 3)

Nama kolom:
['review', 'sentimen', 'translate']


,review,sentimen,translate
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,0,sound and mic satisfied. I only ordered the on...
1,Good. Mudah-mudahan awet. Makasih,1,good. hopefully durable. Thanks
2,"kondisi dus robek2, tolong di perhatikan lagi",0,"the condition of the box is torn, please pay a..."
3,Kualitas produk baik. Harga terjangkau dan pen...,0,product quality is good. affordable prices and...
4,"Overall bagus, tombol berfungsi semuanya, tapi...",0,"overall is good, the buttons all work, but the..."


In [4]:
print("Jumlah seluruh data:", len(df))
print("Jumlah kolom:", len(df.columns))

print("\nDistribusi Sentimen:")
print(df["sentimen"].value_counts())

print("\nPersentase Sentimen:")
print(df["sentimen"].value_counts(normalize=True).mul(100).round(2))

Jumlah seluruh data: 11606
Jumlah kolom: 3

Distribusi Sentimen:
sentimen
0    5803
1    5803
Name: count, dtype: int64

Persentase Sentimen:
sentimen
0    50.0
1    50.0
Name: proportion, dtype: float64


In [5]:
df["case_folding"] = df["review"].astype(str).str.lower()

display(df[["review", "case_folding"]].head(10))

,review,case_folding
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,suara dan mic puas. hanya saya pesan yg nyala ...
1,Good. Mudah-mudahan awet. Makasih,good. mudah-mudahan awet. makasih
2,"kondisi dus robek2, tolong di perhatikan lagi","kondisi dus robek2, tolong di perhatikan lagi"
3,Kualitas produk baik. Harga terjangkau dan pen...,kualitas produk baik. harga terjangkau dan pen...
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall bagus, tombol berfungsi semuanya, tapi..."
5,"Produk ini original, Dan kualitasnya sangat ba...","produk ini original, dan kualitasnya sangat ba..."
6,kualitas sangat jelek. bocor,kualitas sangat jelek. bocor
7,"terimakasih banyak ka ya, barang sudah diterim...","terimakasih banyak ka ya, barang sudah diterim..."
8,"Body nya lebih berat dari cm 1579, spiral nya ...","body nya lebih berat dari cm 1579, spiral nya ..."
9,pesanan di terima tanggal 27 January 2021 teta...,pesanan di terima tanggal 27 january 2021 teta...


In [6]:
def cleaning_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["cleaning"] = df["case_folding"].apply(cleaning_text)

display(df[["case_folding", "cleaning"]].head(10))

,case_folding,cleaning
0,suara dan mic puas. hanya saya pesan yg nyala ...,suara dan mic puas hanya saya pesan yg nyala d...
1,good. mudah-mudahan awet. makasih,good mudahmudahan awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong di perhatikan lagi
3,kualitas produk baik. harga terjangkau dan pen...,kualitas produk baik harga terjangkau dan peng...
4,"overall bagus, tombol berfungsi semuanya, tapi...",overall bagus tombol berfungsi semuanya tapi p...
5,"produk ini original, dan kualitasnya sangat ba...",produk ini original dan kualitasnya sangat bai...
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterim...",terimakasih banyak ka ya barang sudah diterima...
8,"body nya lebih berat dari cm 1579, spiral nya ...",body nya lebih berat dari cm spiral nya lebih ...
9,pesanan di terima tanggal 27 january 2021 teta...,pesanan di terima tanggal january tetapi baru ...


In [7]:
df["tokenisasi"] = df["cleaning"].apply(lambda teks: teks.split())

display(df[["cleaning", "tokenisasi"]].head(10))

,cleaning,tokenisasi
0,suara dan mic puas hanya saya pesan yg nyala d...,"[suara, dan, mic, puas, hanya, saya, pesan, yg..."
1,good mudahmudahan awet makasih,"[good, mudahmudahan, awet, makasih]"
2,kondisi dus robek tolong di perhatikan lagi,"[kondisi, dus, robek, tolong, di, perhatikan, ..."
3,kualitas produk baik harga terjangkau dan peng...,"[kualitas, produk, baik, harga, terjangkau, da..."
4,overall bagus tombol berfungsi semuanya tapi p...,"[overall, bagus, tombol, berfungsi, semuanya, ..."
5,produk ini original dan kualitasnya sangat bai...,"[produk, ini, original, dan, kualitasnya, sang..."
6,kualitas sangat jelek bocor,"[kualitas, sangat, jelek, bocor]"
7,terimakasih banyak ka ya barang sudah diterima...,"[terimakasih, banyak, ka, ya, barang, sudah, d..."
8,body nya lebih berat dari cm spiral nya lebih ...,"[body, nya, lebih, berat, dari, cm, spiral, ny..."
9,pesanan di terima tanggal january tetapi baru ...,"[pesanan, di, terima, tanggal, january, tetapi..."


In [8]:
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

df["stopword_removal"] = df["tokenisasi"].apply(
    lambda tokens: [kata for kata in tokens if kata not in stopwords]
)

display(df[["tokenisasi", "stopword_removal"]].head(10))

,tokenisasi,stopword_removal
0,"[suara, dan, mic, puas, hanya, saya, pesan, yg...","[suara, mic, puas, pesan, yg, nyala, sy, pasti..."
1,"[good, mudahmudahan, awet, makasih]","[good, mudahmudahan, awet, makasih]"
2,"[kondisi, dus, robek, tolong, di, perhatikan, ...","[kondisi, dus, robek, perhatikan]"
3,"[kualitas, produk, baik, harga, terjangkau, da...","[kualitas, produk, baik, harga, terjangkau, pe..."
4,"[overall, bagus, tombol, berfungsi, semuanya, ...","[overall, bagus, tombol, berfungsi, semuanya, ..."
5,"[produk, ini, original, dan, kualitasnya, sang...","[produk, original, kualitasnya, sangat, baik, ..."
6,"[kualitas, sangat, jelek, bocor]","[kualitas, sangat, jelek, bocor]"
7,"[terimakasih, banyak, ka, ya, barang, sudah, d...","[terimakasih, banyak, ka, barang, diterima, ba..."
8,"[body, nya, lebih, berat, dari, cm, spiral, ny...","[body, nya, lebih, berat, cm, spiral, nya, leb..."
9,"[pesanan, di, terima, tanggal, january, tetapi...","[pesanan, terima, tanggal, january, baru, buka..."


In [9]:
factory_stem = StemmerFactory()
stemmer = factory_stem.create_stemmer()

# Mempercepat proses dengan menyimpan hasil kata yang sudah diproses
stem_cache = {}

def stemming_tokens(tokens):
    hasil = []
    for kata in tokens:
        if kata not in stem_cache:
            stem_cache[kata] = stemmer.stem(kata)
        hasil.append(stem_cache[kata])
    return hasil

df["stemming"] = df["stopword_removal"].apply(stemming_tokens)

display(df[["stopword_removal", "stemming"]].head(10))

,stopword_removal,stemming
0,"[suara, mic, puas, pesan, yg, nyala, sy, pasti...","[suara, mic, puas, pesan, yg, nyala, sy, pasti..."
1,"[good, mudahmudahan, awet, makasih]","[good, mudahmudahan, awet, makasih]"
2,"[kondisi, dus, robek, perhatikan]","[kondisi, dus, robek, perhati]"
3,"[kualitas, produk, baik, harga, terjangkau, pe...","[kualitas, produk, baik, harga, jangkau, kirim..."
4,"[overall, bagus, tombol, berfungsi, semuanya, ...","[overall, bagus, tombol, fungsi, semua, packag..."
5,"[produk, original, kualitasnya, sangat, baik, ...","[produk, original, kualitas, sangat, baik, cep..."
6,"[kualitas, sangat, jelek, bocor]","[kualitas, sangat, jelek, bocor]"
7,"[terimakasih, banyak, ka, barang, diterima, ba...","[terimakasih, banyak, ka, barang, terima, baik..."
8,"[body, nya, lebih, berat, cm, spiral, nya, leb...","[body, nya, lebih, berat, cm, spiral, nya, leb..."
9,"[pesanan, terima, tanggal, january, baru, buka...","[pesan, terima, tanggal, january, baru, buka, ..."


In [10]:
hasil_10 = pd.DataFrame({
    "Teks Sebelum": df["review"].head(10),
    "Teks Sesudah": df["stemming"].head(10).apply(
        lambda tokens: " ".join(tokens)
    )
})

display(hasil_10)

,Teks Sebelum,Teks Sesudah
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,suara mic puas pesan yg nyala sy pasti lg klik...
1,Good. Mudah-mudahan awet. Makasih,good mudahmudahan awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati
3,Kualitas produk baik. Harga terjangkau dan pen...,kualitas produk baik harga jangkau kirim cepet...
4,"Overall bagus, tombol berfungsi semuanya, tapi...",overall bagus tombol fungsi semua packaging ku...
5,"Produk ini original, Dan kualitasnya sangat ba...",produk original kualitas sangat baik cepat kir...
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterim...",terimakasih banyak ka barang terima baik sukse...
8,"Body nya lebih berat dari cm 1579, spiral nya ...",body nya lebih berat cm spiral nya lebih tipis...
9,pesanan di terima tanggal 27 January 2021 teta...,pesan terima tanggal january baru buka pagi ta...


## Pengaruh Preprocessing terhadap Kualitas Teks

Preprocessing merupakan tahapan penting dalam pengolahan data teks sebelum dilakukan analisis lebih lanjut. Pada penelitian ini, preprocessing dilakukan melalui beberapa tahapan, yaitu case folding, cleaning, tokenisasi, stopword removal, dan stemming.

Setiap tahapan memberikan pengaruh terhadap kualitas teks, antara lain:

1. **Case Folding:** Mengubah seluruh huruf menjadi huruf kecil sehingga penulisan kata menjadi seragam.
2. **Cleaning:** Menghapus URL, angka, tanda baca, dan karakter khusus yang tidak diperlukan.
3. **Tokenisasi:** Memecah kalimat menjadi kata-kata agar lebih mudah dianalisis.
4. **Stopword Removal:** Menghapus kata-kata umum yang dianggap kurang memberikan informasi penting.
5. **Stemming:** Mengubah kata berimbuhan menjadi kata dasar sehingga bentuk kata menjadi lebih sederhana.

Dengan adanya preprocessing, data ulasan menjadi lebih bersih, terstruktur, dan mudah diolah untuk proses analisis sentimen. Namun, penghapusan angka dan kata tertentu juga dapat menyebabkan sebagian informasi dalam ulasan hilang.